In [0]:
from pyspark.sql import functions as F

VOLUME = "/Volumes/workspace/gs/chegada"
CONTROLE = f"{VOLUME}/_controle"

def ingerir_bronze(fonte):
    """Auto Loader: lê só os arquivos NOVOS da subpasta, guarda como veio
    e acrescenta de onde veio cada linha e quando foi carregada."""
    (spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", f"{CONTROLE}/{fonte}/schema")
        .load(f"{VOLUME}/{fonte}/")
        .withColumn("arquivo_origem", F.col("_metadata.file_path"))
        .withColumn("carregado_em", F.current_timestamp())
        .writeStream
        .option("checkpointLocation", f"{CONTROLE}/{fonte}/checkpoint")
        .trigger(availableNow=True)
        .toTable(f"workspace.gs.bronze_{fonte}")
        .awaitTermination())

In [0]:
for fonte in ["arvores", "clima", "denuncias"]:
    ingerir_bronze(fonte)
    print("ok:", fonte)

ok: arvores
ok: clima
ok: denuncias


In [0]:
%sql
SELECT 'bronze_arvores' AS tabela, COUNT(*) AS linhas, COUNT(DISTINCT arquivo_origem) AS arquivos,
       SUM(json_array_length(items)) AS registros_reais
FROM workspace.gs.bronze_arvores
UNION ALL
SELECT 'bronze_clima', COUNT(*), COUNT(DISTINCT arquivo_origem), COUNT(*)
FROM workspace.gs.bronze_clima
UNION ALL
SELECT 'bronze_denuncias', COUNT(*), COUNT(DISTINCT arquivo_origem), COUNT(*)
FROM workspace.gs.bronze_denuncias

tabela,linhas,arquivos,registros_reais
bronze_arvores,8,8,536
bronze_clima,32,8,32
bronze_denuncias,94,8,94
